# Ch 34 부록 — 한국어 diffusion 붕괴를 직접 복구하기: 80/10/10 마스킹

> 영어 diffusion(Ch 33)을 한국어로 옮길 때(Ch 34), *언어만 바꿨을 뿐* 인데 모델이 유니그램(구두점·고빈도 토큰 반복)으로 **붕괴** 했습니다. step·용량·마스킹 분포를 아무리 바꿔도 소용없었고, 진짜 범인은 **마스킹 방식** — 순진한 *100% `[MASK]`* — 이었습니다. BERT의 고전적 **80/10/10** 트릭이 해결책이었고요. 이 부록은 그 붕괴를 **직접 재현하고 → 진단하고 → 80/10/10으로 고칩니다.**

**이 부록에서 직접 경험할 것**
1. 💥 **붕괴 재현** — 가린 자리를 *전부 `[MASK]`* 로 바꿔 학습 → loss가 유니그램 값에서 평탄, 복원 정확도 바닥
2. 🔬 **진단** — 100% `[MASK]` 는 "입력에 `[MASK]`가 보이면 출력, 아니면 베끼기" 라는 *복사 지름길* 을 열어 줌
3. 🛠️ **수정** — 80/10/10(80% `[MASK]` / 10% 랜덤 토큰 / 10% 원본 유지)으로 그 지름길을 막음 → loss 하락, 복원 회복

**환경**: Google Colab **T4 GPU**. **예상 소요**: 약 18분 (데이터 + 학습 2회를 경량 step으로).

> ⚠️ *경량 재현* 입니다. 완전한 한국어 동화 생성은 30000 step(본 챕터)이 필요하고, 이 부록은 짧은 step으로 *붕괴 vs 정상* 의 **방향**(loss·복원 정확도)만 비교합니다. 마스킹 방식의 차이는 *처음부터* 드러나므로 짧은 학습으로도 충분히 보입니다.

## 0. 환경 셋업

In [ ]:
%pip install -q -U transformers tokenizers datasets accelerate

import math, time, warnings, torch
warnings.filterwarnings("ignore")
from datasets import load_dataset, Dataset

SEED = 42
torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
USE_FP16 = torch.cuda.is_available()
BLOCK = 128
STEPS = 6000          # 경량 — 붕괴 vs 정상의 방향만 (본 챕터는 30000)
print("device:", device, "| fp16:", USE_FP16, "| steps:", STEPS)

# matplotlib 한글 폰트 (Colab — NanumGothic). plot 의 한국어가 □ 로 깨지지 않게.
import matplotlib.pyplot as plt, matplotlib.font_manager as fm, subprocess, os
_fp = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(_fp):
    subprocess.run("apt-get -qq -y install fonts-nanum", shell=True)
fm.fontManager.addfont(_fp)
plt.rcParams["font.family"] = "NanumGothic"
plt.rcParams["axes.unicode_minus"] = False

## 1. 한국어 데이터(TinyStories-Korean) + BBPE 4000

본 챕터와 같은 데이터·토크나이저입니다. 데이터·모델·loss·step 은 두 실험이 똑같이 쓰고, *유일하게 바꾸는 건 콜레이터의 마스킹 방식* 입니다.

In [ ]:
EOT = "<|endoftext|>"
def rebuild(split, n, maxl):
    stories, buf = [], []
    for i, ex in enumerate(load_dataset("g0ster/TinyStories-Korean", split=split, streaming=True)):
        if i >= maxl or len(stories) >= n: break
        line = (ex["text"] or "").strip()
        if line == EOT:
            s = " ".join(buf).strip()
            if s: stories.append(s)
            buf = []
        elif line: buf.append(line)
    return stories[:n]
raw_train = Dataset.from_dict({"text": rebuild("train", 20000, 600_000)})
raw_val   = Dataset.from_dict({"text": rebuild("validation", 500, 50_000)})

from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
from transformers import PreTrainedTokenizerFast
def corpus_iter(bs=1000):
    for i in range(0, len(raw_train), bs): yield raw_train[i:i+bs]["text"]
_tk = Tokenizer(models.BPE(unk_token="[UNK]"))
_tk.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
_tk.decoder = decoders.ByteLevel()
_tk.train_from_iterator(corpus_iter(), trainer=trainers.BpeTrainer(
    vocab_size=4000, special_tokens=["[PAD]", "[UNK]", "[MASK]"],
    initial_alphabet=pre_tokenizers.ByteLevel.alphabet()))
tokenizer = PreTrainedTokenizerFast(tokenizer_object=_tk, pad_token="[PAD]",
                                    unk_token="[UNK]", mask_token="[MASK]")

def tok_fn(b): return {"input_ids": tokenizer(b["text"], add_special_tokens=False)["input_ids"]}
def group(b):
    cat = sum(b["input_ids"], []); n = (len(cat)//BLOCK)*BLOCK
    return {"input_ids": [cat[i:i+BLOCK] for i in range(0, n, BLOCK)]}
def prep(ds): return ds.map(tok_fn, batched=True, remove_columns=ds.column_names).map(group, batched=True)
lm_train = prep(raw_train); lm_val = prep(raw_val)
print(f"vocab {tokenizer.vocab_size} | train chunks {len(lm_train):,} | baseline ln(V)={math.log(4000):.2f}")
print("sample:", raw_train[0]["text"][:80])

## 2. 모델·평가 함수 (본 챕터와 동일, 공통)

작은 `BertForMaskedLM`(hidden 256/4L) + plain CE. 붕괴 실험과 정상 실험이 *똑같은* 모델·loss·step 을 쓰고, 콜레이터만 다릅니다.

In [ ]:
from transformers import BertConfig, BertForMaskedLM, Trainer, TrainingArguments
N_SPECIAL = 3

def new_model():
    cfg = BertConfig(vocab_size=tokenizer.vocab_size, hidden_size=256, num_hidden_layers=4,
                     num_attention_heads=4, intermediate_size=1024,
                     max_position_embeddings=BLOCK, pad_token_id=tokenizer.pad_token_id)
    return BertForMaskedLM(cfg).to(device)

def args_for(tag):
    return TrainingArguments(output_dir=f"./out_{tag}", max_steps=STEPS,
        per_device_train_batch_size=64, learning_rate=5e-4, weight_decay=0.01,
        warmup_steps=500, lr_scheduler_type="cosine", max_grad_norm=1.0, fp16=USE_FP16,
        logging_steps=200, save_strategy="no", report_to="none", remove_unused_columns=False, seed=SEED)

g_eval = torch.Generator().manual_seed(0)
@torch.no_grad()
def fixed_t_acc(model, tv=0.15, n=128):
    model.eval(); cor = tot = 0
    for ex in lm_val.select(range(min(n, len(lm_val)))):
        ids = torch.tensor(ex["input_ids"]); m = torch.rand(len(ids), generator=g_eval) < tv
        if not m.any(): m[0] = True
        inp = ids.clone(); inp[m] = tokenizer.mask_token_id
        pr = model(inp.unsqueeze(0).to(device)).logits[0].argmax(-1).cpu()
        cor += (pr[m] == ids[m]).sum().item(); tot += int(m.sum())
    return cor / tot

@torch.no_grad()
def restore_demo(model, ratio=0.3, k=44):
    # 한 문장을 ratio 만큼 가리고 복원해 본다 (붕괴면 엉뚱, 정상이면 그럴듯)
    g = torch.Generator().manual_seed(1)
    ids = torch.tensor(lm_val[0]["input_ids"][:k])
    m = torch.rand(len(ids), generator=g) < ratio; m[0] = m[0] | (~m.any())
    inp = ids.clone(); inp[m] = tokenizer.mask_token_id
    pr = model(inp.unsqueeze(0).to(device)).logits[0].argmax(-1).cpu()
    out = ids.clone(); out[m] = pr[m]
    return tokenizer.decode(ids), tokenizer.decode(out)
print("준비 완료")

## 3. 💥 시도 1 — 순진한 100% `[MASK]` (영어 레시피 그대로 이식)

영어(Ch 33)에서 잘 됐던 diffusion 콜레이터를 그대로 옮깁니다. 가릴 자리를 고른 뒤, 그 자리를 **전부 `[MASK]`** 로 바꿉니다. 영어에선 이게 잘 됐으니 한국어도 되겠지요.

In [ ]:
class MaskOnlyCollator:
    '''가린 자리를 100% [MASK] 로 — 영어에서 쓰던 순진한 방식.'''
    def __init__(self, tok, eps=0.05, seed=SEED):
        self.mask_id = tok.mask_token_id; self.eps = eps
        self.gen = torch.Generator().manual_seed(seed)
    def __call__(self, ex):
        ids = torch.tensor([e["input_ids"] for e in ex], dtype=torch.long); B, L = ids.shape
        t = torch.rand(B, generator=self.gen) * (1.0 - self.eps) + self.eps
        sel = torch.rand(B, L, generator=self.gen) < t.unsqueeze(1)
        no = ~sel.any(1)
        if no.any(): sel[no, torch.randint(0, L, (int(no.sum()),), generator=self.gen)] = True
        labels = ids.clone(); labels[~sel] = -100
        inp = ids.clone(); inp[sel] = self.mask_id                  # 100% [MASK]
        return {"input_ids": inp, "attention_mask": torch.ones(B, L, dtype=torch.long), "labels": labels}

torch.manual_seed(SEED)
model_bad = new_model()
out_bad = Trainer(model=model_bad, args=args_for("bad"),
                  train_dataset=lm_train, data_collator=MaskOnlyCollator(tokenizer)).train()
acc_bad = fixed_t_acc(model_bad)
print(f"\n[붕괴 시도] train_loss {out_bad.training_loss:.3f}  (baseline ln(V)={math.log(4000):.2f})")
print(f"           fixed-t(0.15) top-1 acc {acc_bad:.3f}")
orig, rest = restore_demo(model_bad)
print(f"원본 : {orig[:70]}")
print(f"복원 : {rest[:70]}")
print("→ loss 가 유니그램 값 근처에서 거의 안 내려가고, 복원도 엉뚱합니다 (유니그램 붕괴).")

## 4. 🔬 진단 — 100% `[MASK]`의 "복사 지름길"

loss 가 baseline(유니그램 엔트로피) 근처에서 평탄하다는 건, 모델이 *문맥* 이 아니라 *빈도* 만 외웠다는 신호입니다. 왜 100% `[MASK]` 가 이걸 부추길까요?

가린 자리를 *전부* `[MASK]` 로 바꾸면, 입력에는 **`[MASK]` 아니면 원본 토큰** 만 존재합니다. 그러면 모델은 위험한 지름길을 배울 수 있습니다 — *"입력이 `[MASK]` 면 뭔가 출력하고, `[MASK]`가 아니면 그 자리 토큰을 그대로 베끼면 된다."* 진짜 문맥 추론 대신에요.

영어에선 토큰이 단어 단위라 이 지름길의 해가 덜했지만, 한국어처럼 토큰이 잘게 쪼개지는(교착어) 환경에선 치명적입니다. 모델이 가린 자리에서 *문맥으로 추론하는 법* 을 아예 안 배우고 유니그램으로 주저앉습니다.

> 그렇다면 데이터·용량 문제가 아니라 *마스킹 방식* 이 범인일까요? BERT(2018)가 바로 이 문제를 알고 있었고, 그 해법이 **80/10/10** 입니다.

## 5. 🛠️ 수정 — 80/10/10 마스킹으로 지름길 막기

BERT의 고전 트릭입니다. 가릴 자리를 고른 뒤, 그 자리를
- **80%** 는 `[MASK]` 로,
- **10%** 는 *임의의 다른 토큰* 으로,
- **10%** 는 *원래 토큰 그대로* 둡니다.

이러면 입력에 보이는 토큰이 `[MASK]` 가 아니어도 *그게 정답이라고 믿을 수 없게* 됩니다(임의로 바뀐 것일 수도 있으니까요). "베끼기" 지름길이 막히고, 모델은 *모든 자리에서 문맥을 봐야만* 합니다. 모델·step·loss 는 그대로, 콜레이터만 바꿉니다.

In [ ]:
class DiffMLMCollator:
    '''가린 자리를 80% [MASK] / 10% 랜덤 / 10% 원본 유지 — BERT 80/10/10.'''
    def __init__(self, tok, eps=0.05, seed=SEED):
        self.mask_id = tok.mask_token_id; self.vocab = tok.vocab_size; self.eps = eps
        self.gen = torch.Generator().manual_seed(seed)
    def __call__(self, ex):
        ids = torch.tensor([e["input_ids"] for e in ex], dtype=torch.long); B, L = ids.shape
        t = torch.rand(B, generator=self.gen) * (1.0 - self.eps) + self.eps
        sel = torch.rand(B, L, generator=self.gen) < t.unsqueeze(1)
        no = ~sel.any(1)
        if no.any(): sel[no, torch.randint(0, L, (int(no.sum()),), generator=self.gen)] = True
        labels = ids.clone(); labels[~sel] = -100
        inp = ids.clone(); r = torch.rand(B, L, generator=self.gen)
        inp[sel & (r < 0.8)] = self.mask_id                              # 80% [MASK]
        rp = sel & (r >= 0.8) & (r < 0.9); nr = int(rp.sum())            # 10% 랜덤
        if nr: inp[rp] = torch.randint(N_SPECIAL, self.vocab, (nr,), generator=self.gen)
        # 10% 원본 유지 (그대로 둠)
        return {"input_ids": inp, "attention_mask": torch.ones(B, L, dtype=torch.long), "labels": labels}

torch.manual_seed(SEED)
model_good = new_model()
out_good = Trainer(model=model_good, args=args_for("good"),
                   train_dataset=lm_train, data_collator=DiffMLMCollator(tokenizer)).train()
acc_good = fixed_t_acc(model_good)
print(f"\n[수정] train_loss {out_good.training_loss:.3f}  (baseline ln(V)={math.log(4000):.2f})")
print(f"       fixed-t(0.15) top-1 acc {acc_good:.3f}")
orig, rest = restore_demo(model_good)
print(f"원본 : {orig[:70]}")
print(f"복원 : {rest[:70]}")
print("→ loss 가 내려가고 복원도 그럴듯해집니다 (8000 step 이라 거칠지만 방향은 분명).")

## 6. 🆚 나란히 비교

In [ ]:
print("=" * 66)
print(f"{'100% [MASK] (붕괴)':<26} loss {out_bad.training_loss:.3f}  acc {acc_bad:.3f}")
print(f"{'80/10/10 (수정)':<26} loss {out_good.training_loss:.3f}  acc {acc_good:.3f}")
print("=" * 66)

import matplotlib.pyplot as plt
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
a1.bar(["100% [MASK]\n(붕괴)", "80/10/10\n(수정)"], [out_bad.training_loss, out_good.training_loss],
       color=["tab:red", "tab:green"], alpha=0.85)
a1.axhline(math.log(4000), ls="--", color="gray", label=f"ln(V)={math.log(4000):.2f}")
a1.set_title("학습 loss (낮을수록 좋음)"); a1.legend()
a2.bar(["100% [MASK]\n(붕괴)", "80/10/10\n(수정)"], [acc_bad, acc_good],
       color=["tab:red", "tab:green"], alpha=0.85)
a2.set_title("고정-t(0.15) 복원 정확도 (높을수록 좋음)")
plt.tight_layout(); plt.show()

## 7. 정리 — 무엇을 배웠나

직접 겪은 순서:

1. **붕괴 재현**: 영어에서 잘 되던 *100% `[MASK]`* diffusion 을 한국어에 그대로 옮기니, loss 가 유니그램 값에서 평탄 (붕괴).
2. **진단**: 100% `[MASK]` 는 "`[MASK]`면 출력, 아니면 베끼기" 라는 *복사 지름길* 을 열어, 모델이 문맥 추론을 안 배움 — 토큰이 잘게 쪼개지는 한국어에서 특히 치명적.
3. **수정**: 80/10/10(80% `[MASK]` / 10% 랜덤 / 10% 원본 유지)으로 그 지름길을 막으니, 모델이 모든 자리에서 문맥을 봐야 해 loss 하락·복원 회복.

**핵심 교훈**
- **"한 가지만 바꿨다"의 함정.** "언어만 바꿨다"고 생각했지만, 영어 레시피를 옮기는 과정에서 *80/10/10이라는 숨은 전제* 가 함께 빠져 있었습니다. 내가 *정말 무엇을 바꿨는지* 를 끝까지 의심해야 합니다.
- **고전이 답일 때가 많습니다.** 80/10/10은 2018년 BERT 논문에 이미 있던 트릭입니다. 새 알고리즘(diffusion)에 옛 지혜가 빠졌던 것뿐입니다.
- 같은 데이터·모델·step·loss 인데 *마스킹 방식 하나* 로 붕괴와 정상이 갈렸습니다. 본 챕터가 80/10/10 을 쓴 이유가 바로 이것입니다.